<a id="optional-fno-ablation"></a>
# 선택 실습 — FNO 푸리에 모드 수와 계산 비용·오차 비교

이 노트북은 [Poisson FNO 필수 실습](../02_Poisson_FNO.ipynb)을 마친 뒤 강사 안내에 따라 진행합니다. 필수 학습과 동시에 실행하지 않습니다.

**실험 질문:** 데이터와 학습 조건을 고정하고 `fno_modes`만 6에서 12로 늘리면 실행 시간, 파라미터 수, GPU 메모리, 테스트 오차가 어떻게 달라질까요?

두 실험은 동일한 축소 데이터셋과 난수 시드, 모델 층 수, 채널 너비, 배치 크기, 200단계 학습을 사용합니다. 바꾸는 요인은 `fno_modes` 하나입니다.


## 1. 환경과 통제 조건 확인

두 설정 파일을 비교해 `fno_modes`와 결과 이름 외의 조건이 같은지 검사합니다.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import subprocess
import sys
import time

launch_dir = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (launch_dir, *launch_dir.parents)
        if (candidate / "labs" / "poisson_fno" / "train_fno.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("labs/poisson_fno/train_fno.py를 찾지 못했습니다.")

LAB_DIR = REPO_ROOT / "labs" / "poisson_fno"
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

import torch
import physicsnemo
import physicsnemo.sym
from notebook_utils import assert_ablation_configs_match, ensure_profile_dataset

config_6 = LAB_DIR / "conf" / "config_FNO_ablation_6.yaml"
config_12 = LAB_DIR / "conf" / "config_FNO_ablation_12.yaml"
differences = assert_ablation_configs_match(config_6, config_12)

print("과정 루트     : {}".format(REPO_ROOT))
print("FNO 실습 폴더 : {}".format(LAB_DIR))
print("GPU           : {}".format(
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CUDA 사용 불가"
))
print("통제한 설정 차이:")
for key, values in differences.items():
    print("  {}: {!r} → {!r}".format(key, values[0], values[1]))


## 2. 실행 전 가설

데이터의 `max_mode=6`과 모델의 `fno_modes=6/12`는 서로 다른 설정입니다. 전자는 데이터에 포함한 최고 주파수, 후자는 모델이 처리하는 푸리에 모드 수입니다.


In [ ]:
# [직접 수정 1] 실행 전에 방향과 근거를 적습니다.
ERROR_HYPOTHESIS = "modes=12의 테스트 오차가 어떻게 달라질지 적으세요."
COST_HYPOTHESIS = "modes=12의 시간·메모리·파라미터 수가 어떻게 달라질지 적으세요."

print("정확도 가설:", ERROR_HYPOTHESIS)
print("계산비용 가설:", COST_HYPOTHESIS)


## 3. 공통 축소 데이터셋 준비

두 실험은 같은 64×64 학습·검증·테스트 데이터를 재사용합니다. 파일 검증을 통과하면 새로 생성하지 않습니다.


In [ ]:
PROFILE_INFO = ensure_profile_dataset(
    LAB_DIR,
    "recovery",
    force=False,
    device="cuda" if torch.cuda.is_available() else "auto",
)
print("공유 데이터셋: {}".format(PROFILE_INFO["dataset_dir"]))


## 4. 모드 6개와 12개를 차례로 실행

전체 실행 시간에는 CUDA 초기화, 파일 입출력, 검증 시간이 포함됩니다. 실행 순서를 무작위화하거나 반복하지 않았으므로 작은 시간 차이를 일반적인 성능 차이로 해석하지 않습니다.


In [ ]:
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
experiments = [
    {"modes": 6, "config": "config_FNO_ablation_6"},
    {"modes": 12, "config": "config_FNO_ablation_12"},
]
RESULTS = []

for experiment in experiments:
    modes = experiment["modes"]
    output_dir = LAB_DIR / "outputs" / "ksc_fno_ablation" / RUN_ID / "modes_{}".format(modes)
    metrics_path = output_dir / "final_state_test_metrics.json"
    command = [
        sys.executable,
        "train_fno.py",
        "--config-name",
        experiment["config"],
        "network_dir={}".format(output_dir),
        "custom.metrics_file={}".format(metrics_path),
    ]

    print("\n" + "=" * 72)
    print("FNO modes={} 실행".format(modes))
    started = time.perf_counter()
    completed = subprocess.run(command, cwd=LAB_DIR)
    wall_seconds = time.perf_counter() - started
    if completed.returncode != 0:
        raise RuntimeError("modes={} 학습 실패 (exit code={})".format(modes, completed.returncode))

    payload = json.loads(metrics_path.read_text(encoding="utf-8"))
    assert payload["profile"] == "recovery"
    assert payload["random_seed"] == 2026
    assert payload["model"]["fno_modes"] == modes
    RESULTS.append(
        {
            "modes": modes,
            "parameters": payload["model"]["trainable_parameters"],
            "wall_seconds": wall_seconds,
            "peak_memory_bytes": payload["runtime_observation"]["peak_memory_allocated_bytes"],
            "relative_l2_before": payload["metrics_before_training"]["relative_l2"],
            "relative_l2_after": payload["metrics_after_training"]["relative_l2"],
            "rmse_after": payload["metrics_after_training"]["rmse"],
        }
    )

print("\n두 실험이 완료되었습니다.")


## 5. 결과 비교

모델 크기·실행 시간·메모리는 계산 비용을 서로 다른 관점에서 보여 줍니다. 상대 L2 오차는 작을수록 예측이 정답에 가깝습니다.


In [ ]:
import matplotlib.pyplot as plt

header = "{:>7} {:>14} {:>12} {:>11} {:>14} {:>14}".format(
    "Modes", "Parameters", "Wall min", "Peak GiB", "L2 before", "L2 after"
)
print(header)
for result in RESULTS:
    peak_gib = (
        result["peak_memory_bytes"] / 2**30
        if result["peak_memory_bytes"] is not None
        else float("nan")
    )
    print("{:7d} {:14,d} {:12.2f} {:11.2f} {:14.6e} {:14.6e}".format(
        result["modes"],
        result["parameters"],
        result["wall_seconds"] / 60.0,
        peak_gib,
        result["relative_l2_before"],
        result["relative_l2_after"],
    ))

low, high = RESULTS
ratios = {
    "parameter_ratio_12_over_6": high["parameters"] / low["parameters"],
    "wall_time_ratio_12_over_6": high["wall_seconds"] / low["wall_seconds"],
    "error_ratio_12_over_6": high["relative_l2_after"] / low["relative_l2_after"],
}
print("\n비율(modes=12 / modes=6)")
for name, value in ratios.items():
    print("  {:32s}: {:.3f}".format(name, value))

labels = ["modes={}".format(result["modes"]) for result in RESULTS]
figure, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
axes[0].bar(labels, [r["wall_seconds"] / 60.0 for r in RESULTS], color=["#76B900", "#1f6feb"])
axes[0].set(title="전체 실행 시간", ylabel="분")
axes[1].bar(labels, [r["parameters"] for r in RESULTS], color=["#76B900", "#1f6feb"])
axes[1].set(title="학습 파라미터 수", ylabel="개")
axes[2].bar(labels, [r["relative_l2_after"] for r in RESULTS], color=["#76B900", "#1f6feb"])
axes[2].set(title="학습 후 테스트 오차", ylabel="상대 L2")
plt.show()


In [ ]:
# [직접 수정 2] 표와 비율을 확인한 뒤 결론을 적습니다.
CONCLUSION = "추가 계산 비용이 오차 변화에 비해 타당했는지 적으세요."
print("실험 결론:", CONCLUSION)


## 6. 해석 범위

1. 모드 12개의 파라미터 수·시간·메모리는 모드 6개의 몇 배입니까?
2. 오차 변화는 추가 계산 비용에 비해 충분히 컸습니까?
3. 데이터의 최고 주파수가 `max_mode=6`이라는 점이 결과에 어떤 영향을 주었습니까?
4. 첫 실행의 CUDA 초기화 비용을 통제하려면 어떤 준비 실행이나 순서 변경이 필요합니까?
5. 여러 난수 시드와 반복 실행의 평균·분산이 있어야 결론을 일반화할 수 있는 이유는 무엇입니까?

이 결과는 지정된 데이터셋, 난수 시드 하나, 200단계 학습, 현재 하드웨어 상태에서 얻은 관찰값입니다.


---

## 참고 자료와 라이선스

공식 문서와 논문 링크는 [PhysicsNeMo 모듈 안내](../README.md#참고-자료)에 모았습니다. 노트북 실행에는 인터넷 연결이 필요하지 않습니다.

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). Existing file-level notices remain in effect.
